In [1]:
# ============================================================
# R1-27  Selection-bias assessment of the 38 paired communities
# Inputs: network_nodes_with_community3.csv , community_porosity.csv
# ============================================================
import pandas as pd, numpy as np
from scipy.stats import mannwhitneyu

# from google.colab import files; files.upload()   # uncomment if needed

nodes = pd.read_csv("network_nodes_with_community3.csv", sep=";", encoding="utf-8-sig")
por   = pd.read_csv("community_porosity.csv")           # comma-delimited

nodes.columns = [c.strip() for c in nodes.columns]
nodes["community"] = pd.to_numeric(nodes["community"], errors="coerce")
size = nodes.groupby("community").size().rename("size")   # node count per community

print("communities in nodes file:", nodes["community"].nunique(),
      "| porosity-file rows:", len(por), "| phases:", por["phase"].unique().tolist())

# communities active in BOTH windows = have a Pre-shift AND a Post-shift row
both = set(por.groupby("community")["phase"].nunique().pipe(lambda s: s[s==2]).index)
pre  = por[por.phase=="Pre-shift"].set_index("community")["porosity"]

sel = sorted(both)
exc = sorted(set(size.index) - both)
sel_size, exc_size = size.reindex(sel), size.reindex(exc)
sel_pre = pre.reindex(sel)
exc_pre = pre.reindex([c for c in exc if c in pre.index])

# ---------- CHECKPOINT vs manuscript ----------
print("\n=== CHECKPOINT (manuscript: n=38, size 7-1075, median 137.5, mean 184.1,",
      "pre-porosity mean 0.141 / median 0.136) ===")
print(f"selected n = {len(sel)}")
print(f"size: min {int(sel_size.min())}  max {int(sel_size.max())}  "
      f"median {sel_size.median()}  mean {sel_size.mean():.1f}")
print(f"pre-shift porosity: mean {sel_pre.mean():.3f}  median {sel_pre.median():.3f}")
print(f"dyads among selected: {int((sel_size==2).sum())}")

# ---------- selection-bias comparison ----------
print("\n=== SELECTED vs EXCLUDED ===")
print(f"selected: n={len(sel)}  median size {sel_size.median()}  "
      f"mean pre-porosity {sel_pre.mean():.3f}")
print(f"excluded: n={len(exc)}  median size {exc_size.median()}  "
      f"dyads {int((exc_size==2).sum())} ({100*(exc_size==2).mean():.1f}%)")
print(f"excluded WITH a pre-shift porosity value: n={len(exc_pre)}  "
      f"mean {exc_pre.mean():.3f}  median {exc_pre.median():.3f}")

Us, ps = mannwhitneyu(sel_size.dropna(), exc_size.dropna(), alternative="greater")
Up, pp = mannwhitneyu(sel_pre.dropna(),  exc_pre.dropna(),  alternative="greater")
print(f"\nMann-Whitney size (selected>excluded):          U={Us:.0f}  p={ps:.2e}")
print(f"Mann-Whitney pre-porosity (selected>excluded):  U={Up:.0f}  p={pp:.2e}")

print(f"\n38 selected hold {int(sel_size.sum())} of {int(size.sum())} nodes "
      f"({100*sel_size.sum()/size.sum():.1f}%), forming "
      f"{100*len(sel)/nodes['community'].nunique():.1f}% of communities")

communities in nodes file: 1667 | porosity-file rows: 194 | phases: ['Pre-shift', 'Post-shift']

=== CHECKPOINT (manuscript: n=38, size 7-1075, median 137.5, mean 184.1, pre-porosity mean 0.141 / median 0.136) ===
selected n = 38
size: min 7  max 1075  median 137.5  mean 184.1
pre-shift porosity: mean 0.141  median 0.136
dyads among selected: 0

=== SELECTED vs EXCLUDED ===
selected: n=38  median size 137.5  mean pre-porosity 0.141
excluded: n=1629  median size 2.0  dyads 1166 (71.6%)
excluded WITH a pre-shift porosity value: n=75  mean 0.006  median 0.000

Mann-Whitney size (selected>excluded):          U=61818  p=4.62e-39
Mann-Whitney pre-porosity (selected>excluded):  U=2573  p=3.13e-17

38 selected hold 6997 of 11426 nodes (61.2%), forming 2.3% of communities
